# e3 · Effect of VIX, Elastic Net (block B3)

Is VIX doing all the work, or do the other macro variables add information? Pairs come from `research_config.BLOCKS["B3"]`.

**Tests.** Each pair is nested (`a` contains `b`'s features), so MSE on log RV uses Clark-West. QLIKE uses Diebold-Mariano, which is undersized for nested models, so read its p-values with caution. For both, the loss differential is averaged across ETFs each month and tested with Newey-West (Andrews bandwidth), as in `garchmidas_phase2.ipynb`. p-values are one-sided, H1: `a` beats `b`.

**Output** in `forecasting/results/`: `e3_losses.csv`, `e3_dm_tests.csv`. `gain_t > 0` favours `a`. `loss_ratio` is `a`'s loss over `b`'s; the Clark-West statistic adjusts for nesting, so it can be significant while the ratio is above 1.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

MODELS_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research_config.py").exists())
sys.path.insert(0, str(MODELS_DIR))
from research_config import BLOCKS, TICKERS

FORECAST_PATH = MODELS_DIR / "forecasting" / "forecasts" / "en_forecasts.csv"
RESULTS_DIR = MODELS_DIR / "forecasting" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

block = BLOCKS["B3"]
df = pd.read_csv(FORECAST_PATH, parse_dates=["date"])
available = set(df["model"])
pairs = [p for p in block["pairs"] if p[1] in available and p[2] in available]
print(f"{len(pairs)} of {len(block['pairs'])} B3 pairs runnable")

In [ ]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where all four variants have a forecast.

In [ ]:
models = [m for m in block["models"] if m in available]
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]
d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"{d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}, {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e3_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model")[LOSSES].round(4)

## Tests

In [ ]:
def one_sided_p(t):
    return float(1 - norm.cdf(t))


loss_idx = losses.set_index(["model", "ETF"])
rows = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"])
    B = d[d["model"] == b].set_index(["date", "etf"])
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    # Clark-West adjusted differential; positive favours a
    cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)

    for etf in ["pooled"] + TICKERS:
        if etf == "pooled":
            dm_t, cw_t = dm_diff.groupby("date").mean(), cw.groupby("date").mean()
        else:
            dm_t, cw_t = dm_diff.xs(etf, level="etf"), cw.xs(etf, level="etf")
        for loss, test, t in [("QLIKE", "DM", -hac_t(dm_t)), ("MSE_log", "Clark-West", hac_t(cw_t))]:
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t),
            })

tests = pd.DataFrame(rows)
tests.to_csv(RESULTS_DIR / "e3_dm_tests.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4)